In [1]:
'''
Fine Tuning
Fine-tuning is a machine learning technique in which a pretrained model, which has
already learned general patterns from a large dataset, is further trained on a smaller,
domain-specific dataset to adapt it for a particular task
'''
%pip install huggingface_hub
%pip install datasets

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
# Loading the yelp_polarity dataset
from datasets import load_dataset

dataset = load_dataset("fancyzhx/yelp_polarity")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})


In [9]:
# Access the first row of the train
train_dataset = dataset['train']
print(train_dataset[0])

{'text': "Unfortunately, the frustration of being Dr. Goldberg's patient is a repeat of the experience I've had with so many other doctors in NYC -- good doctor, terrible staff.  It seems that his staff simply never answers the phone.  It usually takes 2 hours of repeated calling to get an answer.  Who has time for that or wants to deal with it?  I have run into this problem with many other doctors and I just don't get it.  You have office workers, you have patients with medical needs, why isn't anyone answering the phone?  It's incomprehensible and not work the aggravation.  It's with regret that I feel that I have to give Dr. Goldberg 2 stars.", 'label': 0}


In [11]:
# Filtering the yelp_polarity dataset for optimaze performance
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# Get text related for restaurant reviews in train and test datasets
restaurant_train_reviews = train_dataset.filter(
    lambda x: "restaurant" in x["text"].lower()
)

restaurant_test_reviews = test_dataset.filter(
    lambda x: "restaurant" in x["text"].lower()
)

# Shuffle and gets 5000 rows
number_of_reviews = 5000
subset_train_reviews = restaurant_train_reviews.shuffle(seed = 42).select(range(number_of_reviews))
subset_test_reviews = restaurant_test_reviews.shuffle(seed = 42).select(range(number_of_reviews))

# Create a DatasetDict to return both train and test datasets
subset_dataset = {
    "train": subset_train_reviews,
    "test": subset_test_reviews
}

'''
Summary here we will do train model using dataset match with restaurant reviews
'''

Filter: 100%|██████████| 38000/38000 [00:00<00:00, 337924.73 examples/s]


'\nSummary here we will do train model using dataset match with restaurant reviews\n'

In [13]:
# Display the structure to match the requested format
from datasets import DatasetDict

yelp_restaurant_dataset = DatasetDict(subset_dataset)

print(yelp_restaurant_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
})


In [14]:
# Access first rows
yelp_restaurant_dataset['train'][0]

{'text': 'My girlfriend and I have been wanting to come here for awhile, we finally came & we had the worst experience ever. We asked our server for a few minutes to look over the menu & he never came back. 15 minutes later, someone finally came and took our order. We waited awhile and when they brought our food, they got the whole order wrong. My girlfriend ordered soup and it never came out. Worst service ever. Would not recommend this restaurant to anyone.',
 'label': 0}

In [17]:
# Tokenizing the reduced dataset
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Function to tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"],
                    padding = "max_length",
                    truncation = True,
                    max_length = 512)

# Applies the tokenization function to the dataset
tokenized_datasets = yelp_restaurant_dataset.map(
                    tokenize_function,
                    batched = True
                    )

tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
})

In [20]:
# Setting up a pretrained model for sequence classification
from transformers import AutoModelForSequenceClassification
import torch

# Try importing torch_xla for TPU environments
try:
    import torch_xla.core.xla_model as xm
    _xla_available = True
except ImportError:
    _xla_available = False

# Loads a pretrained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint, num_labels = 2
        )

# Determines the device
if _xla_available:
    device = xm.xla_device() # Use XLA device if available
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
            )
# Move the model to the selected device
model.to(device)

'''
Summary here:
- Sequence classification tasks involve assigning a single label or category to an entire sequence of data, such as a sentence, a paragraph, or a longer sequence of tokens.
'''

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14099.45it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [4]:
%pip install transformers[torch]

Defaulting to user installation because normal site-packages is not writeable
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Setting up the TrainingArguments and Trainer classes for fine-tuning
from transformers import Trainer, TrainingArguments

# Setup training arguments
training_args = TrainingArguments(
    output_dir = "./results",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    logging_steps = 500,
    load_best_model_at_end = True,
    optim="adamw_torch", # Explicitly set optimizer to avoid fused=True conflict with xla
)

# Setup the trainer
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets["train"],
    eval_dataset = tokenized_datasets["test"],
)

# Fine-tunes the model
trainer.train()

/opt/conda/envs/HuggingFaceBook/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
# Save model trained in disk
model.to('cpu') # Move model to CPU before saving
model.save_pretrained("./results/final_model")
tokenizer.save_pretrained("./results/final_tokenizer")

'''
Summary here:
- Evaluation loss (0.18255)—The model’s loss during evaluation, with lower values indicating better performance. This value is a relatively low loss value, which generally indicates good performance, especially if you’re working on a classification task such as sentiment analysis.
- Evaluation runtime (63.1454 [seconds])—The total time taken to run the evaluation
- Evaluation samples per second (79.182)—The number of samples processed per second during evaluation. This value reflects the model’s processing speed during evaluation, which seems decent, but it informs you mainly about efficiency rather than quality.
- Evaluation steps per second (4.957)—The number of evaluation steps completed per second. Like evaluation samples per second, this value informs you about efficiency rather than quality.
- Epoch (3.0)—The results from the third epoch of training. You can try varying this value to monitor how the loss evolves with more training.
'''

SyntaxError: incomplete input (1570968994.py, line 6)